# GlucoTwin: Exploratory Data Analysis & Quality Validation
This notebook inspects the harmonized multimodal diabetes datasets across **CGMacros**, **AI-READI Synthetic**, and **HUPA-UCM** cohorts.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

processed_dir = Path("../data/processed")
patients_file = processed_dir / "patients_processed.parquet"
ts_file = processed_dir / "timeseries_processed.parquet"
meals_file = processed_dir / "meals_processed.parquet"
report_file = processed_dir / "data_quality_report.json"

# Fallback to CSV if parquet engines are unavailable
if not patients_file.exists() and (processed_dir / "patients_processed.csv").exists():
    df_patients = pd.read_csv(processed_dir / "patients_processed.csv")
    df_ts = pd.read_csv(processed_dir / "timeseries_processed.csv")
    df_meals = pd.read_csv(processed_dir / "meals_processed.csv")
else:
    df_patients = pd.read_parquet(patients_file)
    df_ts = pd.read_parquet(ts_file)
    df_meals = pd.read_parquet(meals_file)

with open(report_file, "r") as fp:
    quality_report = json.load(fp)

print(f"Total Cohort Patients: {len(df_patients)}")
print(f"Harmonized 5-Min Time-Series Points: {len(df_ts)}")
print(f"Logged Meal Events: {len(df_meals)}")

## 1. Patient Cohort Distribution by Diabetes Status

In [ ]:
status_counts = (
    df_patients.groupby(["source_dataset", "diabetes_status"]).size().unstack(fill_value=0)
)
print("Patient Counts by Dataset and Clinical Status:")
print(status_counts)

status_counts.plot(kind="bar", stacked=True, figsize=(8, 5))
plt.title("Patient Cohort Distribution by Dataset & Status")
plt.xlabel("Source Dataset")
plt.ylabel("Patient Count")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Glycemic Control & Clinical TIR Metrics
- **TIR (Time in Range)**: 70 to 180 mg/dL
- **TAR (Time Above Range)**: > 180 mg/dL
- **TBR (Time Below Range)**: < 70 mg/dL

In [ ]:
valid_glu = df_ts["glucose_mgdl"].dropna()
tir = (valid_glu.between(70, 180)).mean() * 100.0
tar = (valid_glu > 180).mean() * 100.0
tbr = (valid_glu < 70).mean() * 100.0

print(f"Cohort Mean Glucose: {valid_glu.mean():.1f} mg/dL (Std: {valid_glu.std():.1f})")
print(f"Time In Range (70-180 mg/dL): {tir:.1f}%")
print(f"Time Above Range (>180 mg/dL): {tar:.1f}%")
print(f"Time Below Range (<70 mg/dL): {tbr:.1f}%")

plt.figure(figsize=(10, 4))
plt.hist(valid_glu, bins=50, color="teal", edgecolor="black", alpha=0.7)
plt.axvline(70, color="red", linestyle="--", label="Hypoglycemia (<70)")
plt.axvline(180, color="orange", linestyle="--", label="Hyperglycemia (>180)")
plt.axvline(250, color="darkred", linestyle="--", label="Severe Hyper (>250)")
plt.title("CGM Glucose Distribution Across Cohorts")
plt.xlabel("Glucose (mg/dL)")
plt.ylabel("Count")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Meal Carbohydrate Intake Distribution

In [ ]:
if not df_meals.empty:
    print(df_meals[["carbs_g", "protein_g", "fat_g", "calories"]].describe())

    plt.figure(figsize=(8, 4))
    plt.hist(df_meals["carbs_g"].dropna(), bins=30, color="coral", edgecolor="black", alpha=0.7)
    plt.title("Meal Carbohydrate Distribution (Grams)")
    plt.xlabel("Carbs (g)")
    plt.ylabel("Frequency")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. Data Quality & Harmonization Summary

In [ ]:
print(json.dumps(quality_report, indent=2))